# LSTM, GRU: text classification


In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from transformers import TrainingArguments

from core.args import (
    ClassificationDataArguments,
    ClassificationModelArguments,
    ProbeArguments,
)
from core.datasets import build_bpe_tokenizer, build_word_tokenizer, word_tokenize
from core.grad_flow import half_life, probe_grad
from core.model import UniversalRNNClassifier, count_params
from helpers.config import CONFIG_DIR, DATA_DIR, PROCESSED_DIR, parse_config

DATA_PATH = DATA_DIR / "nlp-getting-started" / "train.csv"
OUT_DIR = PROCESSED_DIR / "lecture" / "lstm_gru_classification"
assert DATA_PATH.exists(), f"{DATA_PATH} missing: download via kaggle CLI"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ARG_TYPES = (
    ClassificationModelArguments,
    ClassificationDataArguments,
    ProbeArguments,
    TrainingArguments,
)
runs = [
    parse_config(CONFIG_DIR / f"classification_{m}.yaml", ARG_TYPES)
    for m in ("rnn", "gru", "lstm")
]
data_args = runs[0][1]
assert all(run[1] == data_args for run in runs), "data settings differ across configs"

## Data


In [ ]:
df_twitter = pd.read_csv(DATA_PATH)
df_twitter.sample(5)

## Tokenizers


In [ ]:
PAD_ID = 0

word_counts = Counter()

for t in df_twitter.text.tolist():
    word_counts.update(word_tokenize(t))

print(f"Total unique words: {len(word_counts)}")

# Build vocab: [<pad>, <unk>] + most common words
word_tokenizer = build_word_tokenizer(
    map(word_tokenize, df_twitter.text), max_size=data_args.word_vocab_max
)


def encode_words(text: str):
    return word_tokenizer.convert_tokens_to_ids(word_tokenize(text))


vocab_size_word = len(word_tokenizer)
vocab_size_word

In [ ]:
bpe_tokenizer = build_bpe_tokenizer(df_twitter["text"], data_args.bpe_vocab_size)

# Make PAD_ID=0 across both tokenizers for convenience by remapping if necessary
# (tokenizers library stores vocab as token->id dict; we will just keep PAD as a special pad token externally)
bpe_vocab = bpe_tokenizer.get_vocab()
len(bpe_vocab)

In [ ]:
# Helper wrappers for BPE encode/decode to IDs with PAD=0 fallback
# We'll store PAD as 0 and UNK as the tokenizer's unk id for consistency.
BPE_PAD_ID = 0
BPE_UNK_ID = bpe_tokenizer.unk_token_id


def encode_bpe(text: str):
    ids = bpe_tokenizer.encode(text.lower())
    # Keep original ids; padding will use 0 in collator
    return ids


# Quick smoke test
print(bpe_tokenizer.tokenize("fire at the refinery"))
print(encode_bpe("fire at the refinery")[:10])

## Gradient flow probe


In [ ]:
def plot_curve(lags, norms, label, out_path=None, title=None):
    plt.figure()
    plt.plot(lags, norms, label=label)
    plt.xlabel("Lag s")
    plt.ylabel("Average ||∂·_{t+s} / ∂·_t||_2 (proxy)")
    plt.title(title or label)
    plt.legend()

    if out_path:
        plt.savefig(out_path, bbox_inches="tight")

    plt.show()


def plot_heatmap(heat, out_path=None, title=None):
    plt.figure()
    plt.imshow(heat.numpy(), aspect="auto", origin="lower")
    plt.colorbar(label="||∂·_{t+s} / ∂·_t||_2 (proxy)")
    plt.xlabel("lag s")
    plt.ylabel("time t")

    if title:
        plt.title(title)
    if out_path:
        plt.savefig(out_path, bbox_inches="tight")

    plt.show()

## Load RNN, GRU, LSTM

Trained by `scripts/train_classification.py`.


In [ ]:
results = []
curves = {}

for model_args, _, probe_args, training_args in runs:
    name = model_args.rnn_type.upper()
    out_dir = Path(training_args.output_dir)

    if not (out_dir / "eval_results.json").exists():
        raise FileNotFoundError("Model was not trained yet.")

    model = UniversalRNNClassifier.from_pretrained(out_dir)
    best_auc = json.loads((out_dir / "eval_results.json").read_text())["eval_auc"]

    print(name, "params:", count_params(model))

    probes = probe_grad(
        model, T=probe_args.probe_t, max_lag=probe_args.max_lag, device="cpu"
    )
    for state, (lags, norms, heat) in probes.items():
        label = f"{name} ({state})"
        base = f"{name.lower()}_{state}"
        curves[label] = (lags, norms)

        plot_curve(
            lags,
            norms,
            label,
            out_path=OUT_DIR / f"{base}_curve.png",
            title=f"{label}: gradient vs lag",
        )
        plot_heatmap(
            heat,
            out_path=OUT_DIR / f"{base}_heat.png",
            title=f"{label}: t×lag map",
        )

        results.append(
            {
                "model_name": name,
                "state": state,
                "num_params": count_params(model),
                "best_auc": best_auc,
                "grad_half_life_50pct_lag": half_life(lags, norms, 0.5),
                "grad_half_life_10pct_lag": half_life(lags, norms, 0.1),
            }
        )

df_summary = pd.DataFrame(results)
df_summary.to_csv(OUT_DIR / "summary_grad_flow.csv", index=False)
df_summary

In [ ]:
plt.figure()
for label, (lags, norms) in curves.items():
    plt.plot(lags, norms, label=label)
plt.xlabel("Lag s")
plt.ylabel("Average ||∂·_{t+s} / ∂·_t||_2 (proxy)")
plt.title("Gradient-through-time decay: RNN vs GRU vs LSTM (h & c)")
plt.legend()
plt.savefig(OUT_DIR / "combined_grad_curves.png", bbox_inches="tight")
plt.show()